# 06 — Dashcam Pothole Detector

Train a **separate** detector that works on **dashcam-perspective** footage
(small, distant potholes seen from a windshield), without touching the existing
`models/detector_model.pt` — which stays the detector for the **photo** page.

## Why a second model?
The current detector scores ~85% on its test set but barely fires on real dashcam
video (we measured **2 / 659** frames above conf 0.40, with false positives on
trees/markers). That is a **domain gap**: it was trained mostly on *close-up*
pothole photos, while a dashcam sees potholes small, far away, and at a shallow angle.

## Is this hard work?
- **Extracting frames from videos → easy** (minutes; `src/extract_frames.py`).
- **Training → easy** (your YOLOv8 setup already does it).
- **The real work is *labelling*** (drawing pothole boxes). The smart move is to
  **not label from scratch** — start from a ready-made **road-level pothole dataset
  already in YOLO format** (e.g. Roboflow Universe) and *optionally* add a few of
  your own dashcam frames for domain match.

**Bottom line:** a **half-day, moderate** task. Labelling thousands of your own
frames by hand would be the hard, slow path — we avoid that.

## Output
`models/detector_dashcam.pt` — a 2-class detector `[pothole, traffic_light]`,
a drop-in upgrade for the **Dashboard** (video) page.

## The plan (3 data sources, pick what you need)

| Source | Effort | What it gives |
|---|---|---|
| **A. Road-level YOLO dataset** (recommended) | download (already YOLO) | thousands of *labelled* driving-view potholes — the backbone |
| **B. Your own frames** | extract + label a few hundred | adapts to *your* camera/road/weather |
| **C. Existing traffic data** | 1 cell (reuse) | keeps `traffic_light` working so the model is a full drop-in |

You can train with **A alone** and already beat the current model on video.
B and C are optional improvements.

> **Note on RDD2022:** only its *China_Drone* subset is drone imagery; the rest is
> dashcam/vehicle. But it ships as Pascal-VOC and is country-specific, so we use
> simpler ready-to-go **YOLO-format** datasets instead (see Option A).

In [ ]:
# ── Setup & paths ───────────────────────────────────────────────────────────
import os, shutil, random
from pathlib import Path
import cv2

BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent
print("Project root:", BASE_DIR)

RAW_DIR      = BASE_DIR / "data" / "raw"
DASHCAM_RAW  = RAW_DIR / "dashcam_frames"          # extracted video frames
DATASET_DIR  = BASE_DIR / "data" / "processed" / "dashcam_yolo"   # final YOLO dataset
MODELS_DIR   = BASE_DIR / "models"
EXISTING_DET = BASE_DIR / "data" / "processed" / "detector_yolo"  # for traffic reuse

for d in (RAW_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["pothole", "traffic_light"]   # class 0, class 1
random.seed(42)

## Step 1 — Extract dashcam frames from your videos

Pulls diverse, sharp, de-duplicated frames. Point it at a single video **or** a
folder of videos. This is the *"get dashcam-angle images from dashcam videos"* part.

> Frames alone are **unlabelled** — they become training data only after Step 2.

In [ ]:
# Edit this to your video file or a folder of videos:
VIDEO_SOURCE = str(Path.home() / "Downloads")   # e.g. a single "clip.mp4"

import subprocess, sys
subprocess.run([
    sys.executable, str(BASE_DIR / "src" / "extract_frames.py"),
    VIDEO_SOURCE,
    "-o", str(DASHCAM_RAW),
    "--every", "1.5",          # one frame every 1.5 s
    "--max-per-video", "500",  # cap per video
], check=True)

n = len(list(DASHCAM_RAW.glob("*.jpg")))
print(f"\\n{n} candidate frames in {DASHCAM_RAW}")

## Step 2 — Get labelled potholes

### Option A — a road-level YOLO dataset  *(recommended; least work)*
Grab a **driving/dashcam-view** pothole dataset that is **already in YOLO format**
— no Pascal-VOC conversion needed. Good sources on **Roboflow Universe**
(search `class:pothole`, prefer road-level / dashcam images):

- <https://universe.roboflow.com/search?q=class:pothole>
- e.g. *GeraPotHole*, *Intel Unnati Pothole Detection*, *pothole-detection-yolo-v8*

Download with the Roboflow snippet below (it writes a folder with `images/` +
`labels/`), **or** drop any YOLO dataset folder on disk and point `EXTERNAL_YOLO`
at it. The next cell imports it as the **pothole** class (forced to class 0).

> Skim the images first — pick a dataset shot from a car/road level, not aerial.

### Option B — label your own frames  *(best domain match)*
Upload `data/raw/dashcam_frames/` to **Roboflow** (easiest, web), **CVAT**, or
**LabelImg** (local). Draw one class, `pothole`, export **YOLOv8**. Even 200–500
boxes from *your* camera meaningfully sharpen the model.

### Option C — pseudo-label (speed up B)
Run a model over your frames to *propose* boxes, then only **correct** them in a
tool (Step 2b). Much faster than drawing from scratch.

In [ ]:
# ── Step 2a: Import ready-made YOLO pothole dataset(s) ───────────────────────
# Handles single- AND multi-class datasets: reads each data.yaml, keeps ONLY the
# 'pothole' class, and remaps it to class 0. Keeping multi-class images (covers,
# speed bumps) but labelling only potholes also teaches the model NOT to flag
# manholes as potholes (free hard-negatives).
import yaml
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
POTHOLE_STAGE = RAW_DIR / "pothole_yolo_external"
if POTHOLE_STAGE.exists():
    shutil.rmtree(POTHOLE_STAGE)
(POTHOLE_STAGE / "images").mkdir(parents=True, exist_ok=True)
(POTHOLE_STAGE / "labels").mkdir(parents=True, exist_ok=True)

# List the downloaded YOLO dataset folders (each has data.yaml + train/valid/test).
# Use ONLY dashcam / road-level datasets — skip aerial or steep close-up sets.
EXTERNAL_DATASETS = [
    RAW_DIR / "pothole" / "0-No-dcs-aug.v1i.yolov8",      # dashcam view ✔ (multi-class)
    # RAW_DIR / "pothole" / "Pothole Peddal.v2i.yolov8",  # skipped: close-up, not dashcam
]

def pothole_index(ds_dir: Path) -> int:
    yml = ds_dir / "data.yaml"
    names = yaml.safe_load(yml.read_text()).get("names", []) if yml.exists() else []
    for i, n in enumerate(names):
        if "pothole" in str(n).lower():
            return i
    return 0   # single-class fallback

imported = 0
for ds in EXTERNAL_DATASETS:
    if not ds.exists():
        print(f"  !! missing: {ds}"); continue
    pidx = pothole_index(ds)
    tag = ds.name[:14]
    n_ds = 0
    for split in ("train", "valid", "test"):
        lbl_dir, img_dir = ds / split / "labels", ds / split / "images"
        if not lbl_dir.exists():
            continue
        imgs = {p.stem: p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS}
        for lbl in lbl_dir.glob("*.txt"):
            img = imgs.get(lbl.stem)
            if img is None:
                continue
            out = ["0 " + " ".join(p.split()[1:])
                   for p in lbl.read_text().splitlines()
                   if len(p.split()) == 5 and int(p.split()[0]) == pidx]
            if out:
                stem = f"{tag}_{lbl.stem}"
                shutil.copy2(img, POTHOLE_STAGE / "images" / f"{stem}{img.suffix}")
                (POTHOLE_STAGE / "labels" / f"{stem}.txt").write_text("\\n".join(out))
                n_ds += 1
    print(f"  {ds.name}: pothole class idx={pidx} → kept {n_ds} images")
    imported += n_ds

print(f"\\nImported {imported} pothole images → {POTHOLE_STAGE}")
assert imported > 0, "Nothing imported — check EXTERNAL_DATASETS paths.

In [ ]:
# ── Step 2b (optional): Pseudo-label your extracted frames ───────────────────
# Propose pothole boxes with a baseline model, write YOLO labels to CORRECT later.
# Once you've trained the dashcam model it gives the best proposals; until then
# the existing detector works as a starting point.
from ultralytics import YOLO

PSEUDO_DIR = RAW_DIR / "dashcam_pseudo"
(PSEUDO_DIR / "images").mkdir(parents=True, exist_ok=True)
(PSEUDO_DIR / "labels").mkdir(parents=True, exist_ok=True)

baseline = MODELS_DIR / "detector_dashcam.pt"           # prefer the dashcam model
if not baseline.exists():
    baseline = MODELS_DIR / "detector_model.pt"          # else the existing one
print("Pseudo-labelling with:", baseline.name)

m = YOLO(str(baseline))
PSEUDO_CONF = 0.15      # low recall-first; you delete the wrong ones in the tool
frames = sorted(DASHCAM_RAW.glob("*.jpg"))
written = 0
for f in frames:
    res = m(str(f), conf=PSEUDO_CONF, verbose=False)[0]
    lines = []
    for box in (res.boxes or []):
        cls = int(box.cls[0])
        if cls != 0:        # keep only pothole proposals
            continue
        x, y, w, h = box.xywhn[0].tolist()
        lines.append(f"0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
    if lines:
        shutil.copy2(f, PSEUDO_DIR / "images" / f.name)
        (PSEUDO_DIR / "labels" / f"{f.stem}.txt").write_text("\\n".join(lines))
        written += 1
print(f"Wrote {written} pseudo-labelled frames → {PSEUDO_DIR}")
print("⚠️  These are GUESSES. Open them in Roboflow/LabelImg and fix before training.")

## Step 2c (optional) — reuse existing `traffic_light` data

The current dataset's traffic lights come from **LISA**, which is *already* a
dashcam perspective — that class generalises fine. Reusing it makes the new model
a true **2-class drop-in**, so the Dashboard keeps detecting traffic lights.

Skip this if you only want a pothole-only model (`nc=1`).

In [ ]:
# Pull traffic_light-only labels (class 1) from the existing detector dataset.
TRAFFIC_STAGE = RAW_DIR / "traffic_only_yolo"
(TRAFFIC_STAGE / "images").mkdir(parents=True, exist_ok=True)
(TRAFFIC_STAGE / "labels").mkdir(parents=True, exist_ok=True)

USE_TRAFFIC = True
copied = 0
if USE_TRAFFIC and EXISTING_DET.exists():
    for split in ("train", "val"):
        lbl_dir = EXISTING_DET / "labels" / split
        img_dir = EXISTING_DET / "images" / split
        if not lbl_dir.exists():
            continue
        for lbl in lbl_dir.glob("*.txt"):
            tl_lines = [ln for ln in lbl.read_text().splitlines() if ln.startswith("1 ")]
            if not tl_lines:
                continue
            imgs = list(img_dir.glob(f"{lbl.stem}.*"))
            if not imgs:
                continue
            shutil.copy2(imgs[0], TRAFFIC_STAGE / "images" / imgs[0].name)
            (TRAFFIC_STAGE / "labels" / lbl.name).write_text("\\n".join(tl_lines))
            copied += 1
    print(f"Reused {copied} traffic_light images → {TRAFFIC_STAGE}")
else:
    print("Skipping traffic reuse (USE_TRAFFIC=False or existing dataset missing).")

## Step 3 — Assemble the YOLO dataset

Merge whatever sources you produced (RDD potholes + corrected your-frames +
traffic), shuffle, split 85/15 train/val, and write `dataset.yaml`.

In [ ]:
# ── Build data/processed/dashcam_yolo/{images,labels}/{train,val} ───────────
SOURCES = [POTHOLE_STAGE, PSEUDO_DIR, TRAFFIC_STAGE]   # add/remove as you like
# NOTE: PSEUDO_DIR should contain your *corrected* labels by now.

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
for split in ("train", "val"):
    (DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

pairs = []   # (image_path, label_path)
for src in SOURCES:
    img_dir, lbl_dir = src / "images", src / "labels"
    if not img_dir.exists():
        continue
    for img in img_dir.iterdir():
        lbl = lbl_dir / f"{img.stem}.txt"
        if lbl.exists():
            pairs.append((img, lbl))

random.shuffle(pairs)
n_val = max(1, int(len(pairs) * 0.15)) if pairs else 0
val_set = set(range(n_val))

for i, (img, lbl) in enumerate(pairs):
    split = "val" if i in val_set else "train"
    shutil.copy2(img, DATASET_DIR / "images" / split / img.name)
    shutil.copy2(lbl, DATASET_DIR / "labels" / split / f"{img.stem}.txt")

yaml_text = (
    f"path: {DATASET_DIR.resolve()}\n"
    f"train: images/train\n"
    f"val: images/val\n\n"
    f"nc: {len(CLASS_NAMES)}\n"
    f"names: {CLASS_NAMES}\n"
)
(DATASET_DIR / "dataset.yaml").write_text(yaml_text)
print(f"Dataset: {len(pairs)} labelled images "
      f"({len(pairs)-n_val} train / {n_val} val)")
print(yaml_text)
assert len(pairs) >= 20, "Too few labelled images — import a YOLO dataset (Step 2a) or label more frames.

## Step 4 — Train the dashcam detector

Fine-tunes YOLOv8s. Saves the best weights to **`models/detector_dashcam.pt`**,
leaving `detector_model.pt` (the photo model) untouched.
On CPU this is slow — use a GPU machine if you can. Lower `EPOCHS` for a quick smoke test.

In [ ]:
import torch
from ultralytics import YOLO

EPOCHS = 80
IMGSZ  = 640
device = "0" if torch.cuda.is_available() else "cpu"
print("Training on:", device)

weights = BASE_DIR / "yolov8s.pt"
model = YOLO(str(weights) if weights.exists() else "yolov8s.pt")

model.train(
    data     = str(DATASET_DIR / "dataset.yaml"),
    epochs   = EPOCHS,
    imgsz    = IMGSZ,
    batch    = -1,            # auto
    project  = str(BASE_DIR / "runs"),
    name     = "detector_dashcam",
    exist_ok = True,
    device   = device,
    optimizer= "AdamW",
    lr0      = 0.005,
    cos_lr   = True,
    patience = 20,
    plots    = True,
)

best = BASE_DIR / "runs" / "detector_dashcam" / "weights" / "best.pt"
out  = MODELS_DIR / "detector_dashcam.pt"
if best.exists():
    shutil.copy2(best, out)
    print(f"\\n✅ Saved dashcam detector → {out}")
else:
    print("best.pt not found — check runs/detector_dashcam/weights/")

## Step 5 — Evaluate & eyeball it on real frames

Validate, then run on a few held-out **dashcam** frames to see real boxes.

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt

model = YOLO(str(MODELS_DIR / "detector_dashcam.pt"))
metrics = model.val(data=str(DATASET_DIR / "dataset.yaml"))
print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precision    : {metrics.box.mp:.4f}  Recall: {metrics.box.mr:.4f}")

# Visual check on extracted dashcam frames
sample = sorted(DASHCAM_RAW.glob("*.jpg"))[:6]
if sample:
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    for ax, f in zip(axes.ravel(), sample):
        res = model(str(f), conf=0.25, verbose=False)[0]
        ax.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB))
        ax.set_title(f.name[:24]); ax.axis("off")
    plt.tight_layout(); plt.show()

## Step 6 — Use it in the app

`models/detector_dashcam.pt` is wired with **automatic fallback**:

- The **Dashboard** (video) page requests the **dashcam** model. If
  `detector_dashcam.pt` exists it is used; otherwise it falls back to the
  original detector — so nothing breaks before training finishes.
- The **Detection Pipeline** (photo) page always uses the original
  `detector_model.pt`.

Just restart the backend after training:

```bash
python backend/app.py
```

### To push accuracy further
- Add **more of your own** corrected frames (Step 1–2b) — domain match matters most.
- Train longer / try `yolov8m.pt` if you have GPU headroom.
- Tune the Dashboard confidence (the backend reads `detector_conf` from
  `configs/pipeline_config.yaml`; the dashcam model usually wants ~0.25–0.35).